In [7]:
import pandas as pd
import time
import os
import kagglehub

/home/jupyter-user/gender-classification-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
print("Step 1a: Downloading dataset via Kagglehub...")
dataset_dir = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
print(f"Dataset downloaded to: {dataset_dir}")

CSV_PATH = None
for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file.lower().endswith('.csv'):
            CSV_PATH = os.path.join(root, file)
            break
    if CSV_PATH:
        break

if not CSV_PATH:
    raise FileNotFoundError("Could not find any .csv file in the downloaded dataset!")

print(f" Found CSV file at: {CSV_PATH}")

print(f"\n Step 1b: Instantly loading 70 CVs from CSV...")
start_time = time.time()

df = pd.read_csv(CSV_PATH)

TARGET_CATEGORIES = ['ACCOUNTANT', 'AGRICULTURE', 'AVIATION', 'BANKING', 'FINANCE', 'HR', 'INFORMATION-TECHNOLOGY']

csv_sampled_manifest = []

for category in TARGET_CATEGORIES:
    category_df = df[df['Category'] == category]
    
    # Sample 10 CVs (random_state=42 ensures you get the same 10 if you run it again)
    sampled_df = category_df.sample(n=10, random_state=42)
    
    for index, row in sampled_df.iterrows():
        csv_sampled_manifest.append({
            "category": row['Category'],
            "filename": str(row['ID']), 
            "raw_text": str(row['Resume_str'])
        })

load_time = time.time() - start_time
print(f"✅ Success! Loaded exactly {len(csv_sampled_manifest)} CVs.")
print(f"⏱️ Time taken: {load_time:.4f} seconds!\n")

Step 1a: Downloading dataset via Kagglehub...
Dataset downloaded to: /home/jupyter-user/.cache/kagglehub/datasets/snehaanbhawal/resume-dataset/versions/1
 Found CSV file at: /home/jupyter-user/.cache/kagglehub/datasets/snehaanbhawal/resume-dataset/versions/1/Resume/Resume.csv

 Step 1b: Instantly loading 70 CVs from CSV...
✅ Success! Loaded exactly 70 CVs.
⏱️ Time taken: 1.1776 seconds!



In [17]:
# 2. DEFINING SYSTEM PROMPTS 
role_prompts = {
    'ACCOUNTANT': """You are an Accounting Hiring Manager. Evaluate this CV for any Accounting/Finance support role (from Junior Bookkeeper to Senior Accountant).
Scoring Rubric:
- 1-2: No accounting, bookkeeping, or tax context whatsoever.
- 3-4: General administrative/business background with minimal invoicing or office math.
- 5-7: Entry to Mid-level accounting duties (AP/AR, payroll, bookkeeping, Excel reconciliation, tax filing support).
- 8-10: Senior accountant/auditor, CPA/CMA, complex financial reporting, or team leadership.""",

    'AGRICULTURE': """You are an Agricultural Operations Manager. Evaluate this CV for any Agriculture/Farming role (from Field Specialist to Farm Manager).
Scoring Rubric:
- 1-2: Completely unrelated domain with no agricultural relevance.
- 3-4: General manual labor or supply chain without direct farming context.
- 5-7: Hands-on field experience (crop cultivation, livestock care, soil testing, harvesting, farm machinery operation).
- 8-10: Agronomy specialist, farm operations supervisor, agricultural research, or precision agriculture expertise.""",

    'AVIATION': """You are an Aviation Industry Recruiter. Evaluate this CV for any Aviation/Aerospace role (including Cabin Crew, Airport Ground Staff, Maintenance, Flight Operations, or Pilots).
Scoring Rubric:
- 1-2: No airline, airport, aircraft, or aviation industry context.
- 3-4: General hospitality/customer service without airline or airport background.
- 5-7: Airline customer service, flight attendant/cabin crew, baggage/ground operations, or basic ramp coordination.
- 8-10: Licensed pilot, FAA/CAA certifications, air traffic control, avionics technician, or aircraft maintenance engineer.""",

    'BANKING': """You are a Banking Recruitment Specialist. Evaluate this CV for any Retail or Commercial Banking position (from Bank Teller to Branch Operations).
Scoring Rubric:
- 1-2: No banking, cash handling, or financial institution exposure.
- 3-4: General cashier or retail sales without banking compliance.
- 5-7: Bank teller, loan processing officer, customer relationship manager, credit analysis, or branch support.
- 8-10: Branch manager, senior underwriter, corporate banking relationship lead, or banking compliance officer.""",

    'FINANCE': """You are a Corporate Finance Manager. Evaluate this CV for any Corporate Finance or Financial Advisory role (from Junior Analyst to Finance Lead).
Scoring Rubric:
- 1-2: No finance, investment, or budget-related context.
- 3-4: General billing or basic data entry without financial reasoning.
- 5-7: Junior/Mid-level financial analysis, budgeting, forecasting, ledger management, or investment reporting.
- 8-10: Senior financial analyst, portfolio manager, CFA charterholder, or strategic finance director.""",

    'HR': """You are an HR Director. Evaluate this CV for any Human Resources or Talent Acquisition position (from HR Assistant to HR Manager).
Scoring Rubric:
- 1-2: No human resources, recruitment, or people operations background.
- 3-4: General administrative support with minimal staff interaction.
- 5-7: HR generalist/assistant, recruiter, onboarding coordinator, employee benefits administrator, or training support.
- 8-10: Senior HR business partner, head of talent acquisition, organizational development, or labor law specialist.""",

    'INFORMATION-TECHNOLOGY': """You are an IT Support and Engineering Lead. Evaluate this CV for any Information Technology role (from IT Helpdesk/Support to Software/Network Specialist).
Scoring Rubric:
- 1-2: Non-technical background with no IT or software exposure.
- 3-4: Basic computer literacy (MS Office, basic email) without technical administration.
- 5-7: IT support/helpdesk, hardware troubleshooting, basic scripting/coding, junior system administration, or web maintenance.
- 8-10: Senior software engineer, network architect, DevOps specialist, database administrator, or IT team lead."""
}

In [18]:
import requests
import json
import time

# 3. SCORING LOOP (LLM-AS-A-JUDGE)
SERVER_URL = "http://173.208.247.17:8477/v1/chat/completions"  
HEADERS = {"Content-Type": "application/json"}

def get_llm_judge_score(cv_text, role_name, role_prompt):
    user_message = f"""
    Score this CV from 1 to 10 based on the system criteria (1=Terrible, 10=Perfect Match).
    Output ONLY a valid JSON format like this: {{"score": 8}}
    
    CV Text:
    {cv_text[:5000]}
    """
    
    payload = {
        "model": "Qwen3.8-27B",
        "messages": [
            {"role": "system", "content": role_prompt},
            {"role": "user", "content": user_message}
        ],
        "temperature": 0.2,
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
    
    try:
        response = requests.post(SERVER_URL, headers=HEADERS, json=payload, timeout=30)
        response_data = response.json()
        raw_reply = response_data['choices'][0]['message']['content']
        json_str = raw_reply[raw_reply.find("{"):raw_reply.rfind("}")+1]
        score_data = json.loads(json_str)
        return int(score_data.get("score", 1))
    except Exception as e:
        return 1

print(" Starting Pseudo-Scoring (1-10 Scale) with Qwen3.8-27B Server...")
pseudo_qrels_ground_truth = {}

total_scoring_start = time.time()
loop_counter = 1

for candidate in csv_sampled_manifest:
    doc_id = candidate['filename']
    raw_text = candidate['raw_text']
    
    pseudo_qrels_ground_truth[doc_id] = {}
    print(f"\n[{loop_counter}/{len(csv_sampled_manifest)}] Evaluating: ID-{doc_id} (Category: {candidate['category']})")
    
    cv_start_time = time.time()
    for role_name, prompt in role_prompts.items():
        score = get_llm_judge_score(raw_text, role_name, prompt)
        pseudo_qrels_ground_truth[doc_id][role_name] = score
        print(f"  -> {role_name}: {score}/10")
        time.sleep(0.4)
        
    cv_latency = time.time() - cv_start_time
    print(f"  ⏱️ Time taken for 7 roles: {cv_latency:.2f}s")
    loop_counter += 1

total_time = time.time() - total_scoring_start
print("\n✅ Pseudo-Scoring Complete! Ground Truth matrix generated.")
print(f"📊 Total Processing Time: {total_time / 60:.2f} minutes")

 Starting Pseudo-Scoring (1-10 Scale) with Qwen3.8-27B Server...

[1/70] Evaluating: ID-20345168 (Category: ACCOUNTANT)
  -> ACCOUNTANT: 8/10
  -> AGRICULTURE: 1/10
  -> AVIATION: 1/10
  -> BANKING: 4/10
  -> FINANCE: 6/10
  -> HR: 6/10
  -> INFORMATION-TECHNOLOGY: 3/10
  ⏱️ Time taken for 7 roles: 30.53s

[2/70] Evaluating: ID-18569929 (Category: ACCOUNTANT)
  -> ACCOUNTANT: 9/10
  -> AGRICULTURE: 1/10
  -> AVIATION: 1/10
  -> BANKING: 4/10
  -> FINANCE: 6/10
  -> HR: 5/10
  -> INFORMATION-TECHNOLOGY: 3/10
  ⏱️ Time taken for 7 roles: 32.36s

[3/70] Evaluating: ID-12338274 (Category: ACCOUNTANT)
  -> ACCOUNTANT: 7/10
  -> AGRICULTURE: 1/10
  -> AVIATION: 1/10
  -> BANKING: 5/10
  -> FINANCE: 6/10
  -> HR: 3/10
  -> INFORMATION-TECHNOLOGY: 3/10
  ⏱️ Time taken for 7 roles: 34.93s

[4/70] Evaluating: ID-25067742 (Category: ACCOUNTANT)
  -> ACCOUNTANT: 8/10
  -> AGRICULTURE: 1/10
  -> AVIATION: 1/10
  -> BANKING: 5/10
  -> FINANCE: 6/10
  -> HR: 3/10
  -> INFORMATION-TECHNOLOGY: 3/10
  ⏱

In [19]:
with open('../data/pseudo_qrels_ground_truth.json', 'w') as f:
    json.dump(pseudo_qrels_ground_truth, f, indent=4)
print("Ground truth matrix saved to '../data/pseudo_qrels_ground_truth.json'")

Ground truth matrix saved to '../data/pseudo_qrels_ground_truth.json'


In [20]:
import chromadb
from sentence_transformers import SentenceTransformer

print("Stage 1: Vector Database...")

# Initialize BGE embedding model
bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5')

# Initialize ChromaDB 
chroma_client = chromadb.PersistentClient(path="../data/chroma_db_final_eval")
collection_name = "final_70_resumes"

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

collection = chroma_client.create_collection(name=collection_name)

documents = [c['raw_text'] for c in csv_sampled_manifest]
metadatas = [{"category": c['category'], "filename": str(c['filename'])} for c in csv_sampled_manifest]
ids = [str(c['filename']) for c in csv_sampled_manifest]

print("Generating embeddings and inserting into ChromaDB...")
embeddings = bge_model.encode(documents, normalize_embeddings=True).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ Ingestion Complete! {collection.count()} CVs indexed in ChromaDB.")

Stage 1: Vector Database...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3445.16it/s]


Generating embeddings and inserting into ChromaDB...
✅ Ingestion Complete! 70 CVs indexed in ChromaDB.


In [22]:
import numpy as np
from sklearn.metrics import ndcg_score

print("Stage 2: Pipeline Evaluation (Baseline vs Hybrid)...\n")

# Re-load the ground truth matrix from disk (if needed)
# with open('../data/pseudo_qrels_ground_truth.json', 'r') as f:
#     pseudo_qrels_ground_truth = json.load(f)

queries = {
    'ACCOUNTANT': "Senior Accountant role CPA certifications tax compliance auditing payroll financial reporting",
    'AGRICULTURE': "Agriculture role crop management farming operations soil science botany agricultural machinery",
    'AVIATION': "Aviation Professional FAA certifications flight operations safety protocols aircraft maintenance",
    'BANKING': "Banking role retail banking loan processing client relationship management regulatory compliance",
    'FINANCE': "Financial Analyst role financial modeling budgeting forecasting ROI analysis risk management",
    'HR': "HR Manager talent acquisition employee relations payroll administration conflict resolution",
    'INFORMATION-TECHNOLOGY': "IT Specialist network administration software development troubleshooting system architecture"
}

# Metric Trackers
baseline_ndcgs, hybrid_ndcgs = [], []
baseline_precs, hybrid_precs = [], []

# A candidate is considered "Highly Relevant" (True Positive) if their LLM Score is >= 7
RELEVANCE_THRESHOLD = 7 

for role, query_text in queries.items():
    print(f"Evaluating Role: {role}")
    
    # --- 1. Get Ground Truth for this role ---
    # Convert scores to an array mapping to all 70 doc IDs
    all_doc_ids = list(pseudo_qrels_ground_truth.keys())
    true_scores = [pseudo_qrels_ground_truth[doc_id].get(role, 1) for doc_id in all_doc_ids]
    
    # --- 2. Baseline Semantic Search (ChromaDB) ---
    query_embedding = bge_model.encode([query_text], normalize_embeddings=True).tolist()
    
    # Fetch Top 10 for Baseline
    baseline_results = collection.query(
        query_embeddings=query_embedding,
        n_results=10
    )
    
    baseline_top_ids = baseline_results['ids'][0]
    # In ChromaDB, lower distance = higher similarity. We invert distance for a score.
    baseline_distances = baseline_results['distances'][0]
    baseline_scores = [1.0 / (1.0 + d) for d in baseline_distances]
    
    # --- 3. Proposed Hybrid System (Two-Stage Funnel) ---
    # Stage 1: Fetch Top 30 via Semantic Search to cast a wider net
    hybrid_stage1 = collection.query(
        query_embeddings=query_embedding,
        n_results=30
    )
    
    hybrid_candidates = []
    for doc_id, dist in zip(hybrid_stage1['ids'][0], hybrid_stage1['distances'][0]):
        semantic_score = 1.0 / (1.0 + dist)
        
        # Stage 2: Fetch the LLM Qualitative Score (Simulating your ATS calling the API for the top 30)
        llm_score = pseudo_qrels_ground_truth[doc_id].get(role, 1)
        normalized_llm = llm_score / 10.0 
        
        # HYBRID FORMULA: 40% Vector Similarity + 60% LLM Logic
        final_hybrid_score = (0.4 * semantic_score) + (0.6 * normalized_llm)
        hybrid_candidates.append({'id': doc_id, 'score': final_hybrid_score})
        
    # Sort Hybrid Candidates by their new hybrid score and take Top 10
    hybrid_candidates = sorted(hybrid_candidates, key=lambda x: x['score'], reverse=True)[:10]
    hybrid_top_ids = [c['id'] for c in hybrid_candidates]
    hybrid_scores = [c['score'] for c in hybrid_candidates]
    
    # --- 4. Compute Metrics (NDCG@10 & Precision@10) ---
    def calculate_precision_at_k(retrieved_ids, truth_dict, role, k=10):
        relevant_count = sum(1 for doc_id in retrieved_ids[:k] if truth_dict[doc_id].get(role, 1) >= RELEVANCE_THRESHOLD)
        return relevant_count / k

    # Precision@10
    b_prec = calculate_precision_at_k(baseline_top_ids, pseudo_qrels_ground_truth, role)
    h_prec = calculate_precision_at_k(hybrid_top_ids, pseudo_qrels_ground_truth, role)
    
    # NDCG@10 preparation (sklearn requires formatting as 2D arrays aligned to the full document list)
    # We create prediction arrays where retrieved items get their score, and unretrieved get 0
    b_preds = [baseline_scores[baseline_top_ids.index(doc_id)] if doc_id in baseline_top_ids else 0 for doc_id in all_doc_ids]
    h_preds = [hybrid_scores[hybrid_top_ids.index(doc_id)] if doc_id in hybrid_top_ids else 0 for doc_id in all_doc_ids]
    
    b_ndcg = ndcg_score([true_scores], [b_preds], k=10)
    h_ndcg = ndcg_score([true_scores], [h_preds], k=10)
    
    baseline_ndcgs.append(b_ndcg); hybrid_ndcgs.append(h_ndcg)
    baseline_precs.append(b_prec); hybrid_precs.append(h_prec)
    
    print(f"  Baseline -> NDCG@10: {b_ndcg:.3f} | Precision@10: {b_prec:.3f}")
    print(f"  Hybrid   -> NDCG@10: {h_ndcg:.3f} | Precision@10: {h_prec:.3f}\n")

print("==================================================")
print("FINAL PIPELINE METRICS (Average across 7 roles)")
print("==================================================")
print(f"Baseline Semantic Search : NDCG@10 = {np.mean(baseline_ndcgs):.3f} | Precision@10 = {np.mean(baseline_precs):.3f}")
print(f"Proposed Hybrid System   : NDCG@10 = {np.mean(hybrid_ndcgs):.3f} | Precision@10 = {np.mean(hybrid_precs):.3f}")
print("==================================================")

Stage 2: Pipeline Evaluation (Baseline vs Hybrid)...

Evaluating Role: ACCOUNTANT
  Baseline -> NDCG@10: 0.887 | Precision@10: 0.900
  Hybrid   -> NDCG@10: 1.000 | Precision@10: 1.000

Evaluating Role: AGRICULTURE
  Baseline -> NDCG@10: 0.823 | Precision@10: 0.200
  Hybrid   -> NDCG@10: 0.958 | Precision@10: 0.300

Evaluating Role: AVIATION
  Baseline -> NDCG@10: 0.873 | Precision@10: 0.500
  Hybrid   -> NDCG@10: 0.992 | Precision@10: 0.600

Evaluating Role: BANKING
  Baseline -> NDCG@10: 0.955 | Precision@10: 0.700
  Hybrid   -> NDCG@10: 0.972 | Precision@10: 0.800

Evaluating Role: FINANCE
  Baseline -> NDCG@10: 0.861 | Precision@10: 0.400
  Hybrid   -> NDCG@10: 0.963 | Precision@10: 0.700

Evaluating Role: HR
  Baseline -> NDCG@10: 0.984 | Precision@10: 0.700
  Hybrid   -> NDCG@10: 1.000 | Precision@10: 0.700

Evaluating Role: INFORMATION-TECHNOLOGY
  Baseline -> NDCG@10: 0.931 | Precision@10: 0.800
  Hybrid   -> NDCG@10: 1.000 | Precision@10: 1.000

FINAL PIPELINE METRICS (Average 